In [ ]:
import torch
import matplotlib.pyplot as plt

from palmprint.models.siamese import SiameseNetwork
from palmprint.training.train import load_checkpoint
from palmprint.datasets.bmpd import build_bmpd_splits
from palmprint.sampling.pairs import generate_balanced_pairs
from palmprint.datasets.pair_dataset import PairDataset
from palmprint.utils.seed import set_seed
from sklearn.metrics import roc_curve, auc

from torch.utils.data import DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SiameseNetwork(
    embedding_dim=128,
    input_channels=1,
)

checkpoint = load_checkpoint(
    model,
    "../checkpoints/siamese_bmpd_roi_best.pt",
    device,
)

model.eval()

In [ ]:
splits = build_bmpd_splits(
    DATA_ROOT,
    seed=42,
    train_ratio=0.7,
    val_ratio=0.1,
    test_ratio=0.2,
)

In [ ]:
test_pairs = generate_balanced_pairs(
    splits["test"],
    num_pairs=10000,
    seed=42,
)

print(f"Train pairs: {len(test_pairs)}")

In [ ]:
test_dataset = PairDataset(
    test_pairs,
    image_size=224,
    grayscale=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

In [ ]:
with torch.no_grad():
    for batch in test_loader:
        img1, img2, labels = batch
        img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
        outputs = model(img1, img2)
        print(outputs.shape)
        break

In [ ]:
def plot_roc(self):
    if self.y_scores is None:
        print("No scores provided for ROC curve.")
        return

    fpr, tpr, _ = roc_curve(self.y_true, self.y_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle='--')  # random baseline
    plt.xlabel("False Positive Rate (FAR)")
    plt.ylabel("True Positive Rate (TAR)")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
def plot_far_frr(self):
    if self.y_scores is None:
        print("No scores provided.")
        return

    fpr, tpr, thresholds = roc_curve(self.y_true, self.y_scores)
    fnr = 1 - tpr

    plt.figure()
    plt.plot(thresholds, fpr, label="FAR")
    plt.plot(thresholds, fnr, label="FRR")
    plt.xlabel("Threshold")
    plt.ylabel("Error Rate")
    plt.title("FAR vs FRR")
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
def plot_score_distribution(self):
    if self.y_scores is None:
        print("No scores provided.")
        return

    genuine = self.y_scores[self.y_true == 1]
    imposter = self.y_scores[self.y_true == 0]

    plt.figure()
    plt.hist(genuine, bins=50, alpha=0.5, label="Genuine")
    plt.hist(imposter, bins=50, alpha=0.5, label="Imposter")
    plt.xlabel("Score")
    plt.ylabel("Frequency")
    plt.title("Score Distribution")
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
metrics = PalmprintMetrics(
    y_true=labels,
    y_pred=preds,
    y_scores=-distances 
)

metrics.report()

metrics.plot_roc()
metrics.plot_far_frr()
metrics.plot_score_distribution()